# DeepLabCut 工具箱 - 演示（鼠标抓取动作）

以下是一些可以提供的有用资源：

- [github.com/DeepLabCut/DeepLabCut](https://github.com/DeepLabCut/DeepLabCut)
- [DeepLabCut 文档：单动物项目用户指南](https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html)

#### 本 Jupyter Notebook 伴随以下用户指南：

Nath\*, Mathis\* 等人。*Using DeepLabCut for markerless pose estimation during behavior across species* Nature Protocols, 2019（使用 DeepLabCut 进行跨物种行为过程中的无标记姿态估计）：https://www.nature.com/articles/s41596-019-0176-0

本 Notebook 从一个已经初始化并标注过数据的项目开始。

**数据来源：** 数据集来自 Mathis\* 等人。*Somatosensory Cortex Plays an Essential Role in Forelimb Motor Adaptation in Mice* Neuron, 2017（体感皮层在小鼠前肢运动适应中起关键作用）：DOI:https://doi.org/10.1016/j.neuron.2017.02.049

本文档（Notebook）演示了如何执行以下操作：
- 绘制已标注的图像
- 训练一个模型（网络）
- 评估一个模型（网络）
- 分析一个新的视频
- 创建一个自动标注的视频
- 绘制轨迹
- 识别异常帧
- 手动注释异常帧
- 合并数据集并更新训练集
- 训练一个模型（网络）

## 导入工具箱和必需的库

In [ ]:
from pathlib import Path

import deeplabcut

### 设置一个变量指向项目配置文件：

In [ ]:
# Create a variable to set the config.yaml file path:
# If this path does not point to the project from the URL below,
# edit it to make sure it does:
#   https://github.com/DeepLabCut/DeepLabCut/tree/main/examples/Reaching-Mackenzie-2018-08-30
# 
# Example - Linux/OSX
#   path_config_file = "/Users/john/DeepLabCut/examples/Reaching-Mackenzie-2018-08-30/config.yaml"
# Example - Windows
#   path_config_file = r"C:\DeepLabCut\examples\Reaching-Mackenzie-2018-08-30\config.yaml"

path_config_file = str(Path.cwd() / "Reaching-Mackenzie-2018-08-30" / "config.yaml")
print(path_config_file)

**注意**: 当您在自己的数据上使用 DeepLabCut 时，您需要执行以下步骤：(1) 创建一个项目，(2) 提取需要标注的帧，以及 (3) 进行数据标注。
**在本演示中，所有这些步骤都已为您完成！** 本演示的目的是让您熟悉工作流程的某一部分。

### 加载预标注数据：

In [ ]:
# Let's load some demo data, and create a training set 
# (note, this function is not used when you create your own project):

deeplabcut.load_demo_data(path_config_file)

In [ ]:
# Perhaps plot the labels to see how the frames were annotated:

deeplabcut.check_labels(path_config_file)

## 开始训练特征检测器

此函数会针对训练数据集的特定洗牌（shuffle）来训练神经网络。**用户可以在 `.../Reaching-Mackenzie-2018-08-30/dlc-models-pytorch/iteration-0/ReachingAug30-trainset95shuffle1/train/pytorch_config.yaml` 中设置各种参数**。有关可设置变量的更多信息，请查阅 [文档](https://deeplabcut.github.io/DeepLabCut/docs/pytorch/pytorch_config.html)！

训练可以随时停止。请注意，权重仅在每经过 'save\_epochs' 步时才会保存。对于此演示，建议非常频繁地保存和显示进度（例如，每 20 步显示一次，每 2 步保存一次）。在实际应用中，这样做效率很低（在实际训练中，您会训练到约 200 步，因此我们每 10 步保存一次）。

**我们建议只训练 15-20 分钟，因为您运行此演示的目的不是为了使用 DLC（DeepLabCut），而只是为了走完这些步骤。总的来说，此演示花费的时间应该少于 1 小时！**

In [ ]:
# notice the variables "save_epochs" and "displayiters" that can be set in the function
deeplabcut.train_network(path_config_file, shuffle=1, save_epochs=2, displayiters=10)

# you just need to run this until you get at least 1 snapshot, which is set by: "save_epochs" 
# (so in this case you could stop after 2 epochs!) How do I stop? Click the STOP button!

# To train until ~50 epochs on a CPU should be ~15 min
# Every 10 epochs, your model will be evaluated. You can keep an eye on model performance
# while the model is being trained.

*请注意，如果您通过“stop”命令或按下 CTRL+C 来停止它，您会看到一个键盘中断“错误”，但这并不是一个真正的错误，即您可以忽略它。*

## 评估训练好的网络

此函数用于在特定的训练状态（快照）或所有状态下，对一个已训练好的模型进行特定洗牌（shuffle/shuffles）的评估。网络将在数据集（图像）上进行评估，并将结果以 `.csv` 文件的形式存储在 `evaluation-results-pytorch` 目录下的一个子目录中。

你可以在本项目中的 `config.yaml` 文件中更改各种参数。在进行评估时，可以更改 `pcutoff`。这个“截止值”（cutoff）也影响了估计位置的概率需要多高，才会在图表中显示出来。

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

**注意：根据您的具体设置，有时您可能会遇到一些 "matplotlib 错误，但这些错误并不重要**

现在您可以去查看生成的图像了。鉴于输入的数据量有限，并且我们花费了大约 20 分钟进行测试，因此这些图像的拟合效果不会很理想，所以请不必担心。这只是为了让您熟悉整个工作流程……

## 分析视频

此功能根据**已训练的网络模型**从视频中提取姿态信息。用户可以选择要使用的预训练模型——默认情况下，系统会使用最新的快照（snapshot）来分析视频。但是，用户也可以在 `config.yaml` 文件中指定 `snapshotindex` 变量的值来确定要使用的快照索引。

分析结果将存储在与视频位于**同一目录**下的 `.hd5` 文件中。姿态数组（即姿态相对于帧索引的映射）也可以导出为 `.csv` 文件（通过设置相关标志来实现...）。

In [ ]:
# Set the video path:
# The video can be the one you trained with and new videos that look similar, i.e. same experiments, etc.
# You can add individual videos, OR just a folder - it will skip videos that are already analyzed once.

# i.e. you can run 'reachingvideo1' and/or 'MovieS2_Perturbation_noLaser_compressed'

videofile_path = str(Path(path_config_file).parent / "videos" / "reachingvideo1.avi")

In [ ]:
print("Start Analyzing the video!")

deeplabcut.analyze_videos(path_config_file, [videofile_path])
# this video takes ~ 1 min to analyze with a CPU

*注意：是的，这在 CPU 上运行很慢（GPU 的速度要快得多）。如果您感兴趣，请参阅 https://www.biorxiv.org/content/early/2018/10/30/457242！*

## 创建带标签的视频

此函数用于可视化目的，可用于创建包含预测标签的 `.mp4` 格式视频。此视频将保存在与（未标记）视频相同的目录中。

可以设置与颜色映射（colormap）和点大小（dotsize）相关的各种参数（后端使用 matplotlib）。有关如何设置这些参数的详细信息，请参阅 `config.yaml` 文件。

In [ ]:
deeplabcut.create_labeled_video(path_config_file, [videofile_path], draw_skeleton=True)

## 绘制分析视频的轨迹

此函数会绘制整个视频中所有身体部位的轨迹。每个身体部位都由一个唯一的颜色标识。底层的函数可以轻松地进行自定义设置。

In [ ]:
%matplotlib notebook
deeplabcut.plot_trajectories(path_config_file, [videofile_path], showfigures=True)

# These plots are interactive and can be customized (see https://matplotlib.org/)

## 提取预测结果出现偏差的异常帧

这是一个可选步骤，允许在评估结果不佳时增加更多的训练数据。在这种情况下，用户可以使用以下函数来提取标签被错误预测的帧。请确保提供 `"iterations"` 的正确值，因为它将用于创建保存所提取帧的唯一目录。

In [ ]:
# Note, if you have questions on parameters, remember "?" gives you answers:
deeplabcut.extract_outlier_frames?

In [ ]:
deeplabcut.extract_outlier_frames(
    path_config_file,
    videofile_path,
    outlieralgorithm="uncertain",
    p_bound=0.2,
)

用户可以**迭代式地**运行此过程，甚至可以从同一视频中提取**额外的帧**。

## 手动修改标签

此步骤允许用户修正从视频中提取出来的标签。请导航到包含视频的文件夹，并按照协议描述使用 **GUI** 来更新标签。

关于 **GUI** 的文档，请参阅 [`napari-deeplabcut`](https://github.com/DeepLabCut/napari-deeplabcut/tree/main) 的文档——特别是关于 _"3. Refining labels – the image folder contains a machinelabels-iter<#>.h5 file."_ 的部分！

In [ ]:
deeplabcut.refine_labels(path_config_file)

In [ ]:
# Now merge datasets (once you refined all frames)
deeplabcut.merge_datasets(path_config_file)

## 创建一个新的训练数据集迭代，检查它并开始训练...

根据完善后的标签，将这些帧追加到原始数据集中以创建一个新的训练数据集迭代。

In [ ]:
#Perhaps plot the labels to see how how all the frames are annotated (including the refined ones)
deeplabcut.check_labels(path_config_file)
# if they are off, you can load them in the labeling_gui to adjust!

In [ ]:
deeplabcut.create_training_dataset(path_config_file, engine=deeplabcut.Engine.PYTORCH)

现在一个可以再次训练网络了...（使用扩展的数据集）。 我们可以通过使用 `snapshot_path` 参数，在已有的快照（snapshot）基础上继续训练——而不是从头开始训练模型，它会加载我们已经拥有的权重并对其进行微调！

In [ ]:
snapshot_path = (  # Edit me if needed! Select the path to the snapshot to continue training from!
    Path(path_config_file).parent / 
    "dlc-models-pytorch" / 
    "iteration-0" / 
    "ReachingAug30-trainset95shuffle1" / 
    "train" / 
    "snapshot-best-080.pt"
)

deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    save_epochs=2,
    displayiters=10,
    batch_size=8,
    snapshot_path=snapshot_path,
)